(course-core-12)=

# Module 12: Building, Repairing and Auditing Systems

**Welcome back, Apprentice Master.** In [Module 11: Iterating Systems](../Common_Core/11_Iterating_Systems.ipynb), you mastered memory-efficient streaming with `msm.Iterator()`. Up to this point in the course, all functions you have used (`msm.convert()`, `msm.info()`, `msm.select()`, `msm.get()`, `msm.set()`, `msm.Iterator()`) belong to the core **`msm.basic`** module (aliased directly under top-level `msm.*`).

Now we step into specialized function modules by exploring **`msm.build`**: MolSysMT's workshop for system preparation, editing, reconstruction, synthesis, and quality auditing.

### Functional Pillars of `msm.build`  

The **`msm.build`** module encompasses three major scientific pillars:
1. **Audit and Inspection**: `get_non_standard_residues()`, `get_missing_residues()`, `get_missing_heavy_atoms()`, `get_missing_bonds()`, `get_disulfide_bonds()`, `get_missing_terminal_cappings()`, `has_hydrogens()`, `is_solvated()`.
2. **Repair and Completion**: `add_missing_heavy_atoms()`, `add_missing_hydrogens()`, `add_missing_bonds()`, `add_missing_terminal_cappings()`, `solve_atoms_with_alternate_location()`.
3. **Synthesis and Solvation**: `build_peptide()`, `solvate()`, `make_water_box()`, `mutate()`, `make_bioassembly()`, `remove_overlapping_molecules()`.

```{admonition} Glossary: MolSysMT modules
:class: dropdown info
MolSysMT organizes specialized capabilities into domain-specific function modules:
- **`msm.basic`**: Core form-agnostic operations (conversion, selection, attribute extraction, modification, streaming).
- **`msm.build`**: System editing, residue/atom addition, solvation, synthesis, and quality auditing.
- **`msm.topology`**: Sequence alignment, covalent graphs, and secondary structure.
- **`msm.structure`**: Spatial alignment, RMSD, contacts, and geometric transformations.
- **`msm.physchem`**: Physical and chemical properties (charge, mass, SASA).

This module introduces **`msm.build`**, focusing on how to audit structural completeness, perform system repairs, and synthesize peptides from sequence.
```

(course-core-12-learning-outcomes)=
```{admonition} Learning Outcomes
:class: dropdown learning-outcomes

By the end of this module, you will be able to:
- Navigate the functional scope of the `msm.build` module.
- Identify non-standard groups, sequence gaps, and missing hydrogens using `msm.build.get_non_standard_residues()` and `msm.build.has_hydrogens()`.
- Repair incomplete systems by adding missing hydrogens with `msm.build.add_missing_hydrogens()`.
- Audit missing covalent connectivity and reconstruct bonds using `msm.build.get_missing_bonds()` and `msm.build.add_missing_bonds()`.
- Synthesize new peptide structures directly from amino acid sequences using `msm.build.build_peptide()`.
```

### 1. Auditing Anomalies

Let's begin by importing MolSysMT and loading our T4 Lysozyme demonstration system.

In [1]:
import molsysmt as msm
from molsysmt import systems

# Load T4 Lysozyme system as native MolSys
lysozyme = msm.convert(systems['T4 lysozyme L99A']['181l.bcif.gz'], to_form='molsysmt.MolSys')

Experimental PDB and mmCIF files contain co-crystallized ligands, non-standard amino acids, or unobserved flexible loops. You can audit non-standard residues and hydrogen status using `msm.build` inspection functions:

In [2]:
# Audit non-standard groups in T4 Lysozyme
non_std_groups = msm.build.get_non_standard_residues(lysozyme)
print(f"Non-standard groups detected: {non_std_groups}")

# Check if the system contains hydrogen atoms
has_h = msm.build.has_hydrogens(lysozyme)
print(f"Does the system contain hydrogen atoms? {has_h}")

Non-standard groups detected: {}
Does the system contain hydrogen atoms? False


:::{hint}
:class: dropdown
**msm.build.get_non_standard_residues()**: Scans a molecular system topology to identify non-standard amino acids, modified nucleotides, or co-crystallized ligands. See API doc: {func}`molsysmt.build.get_non_standard_residues`.
:::

### 2. Repairing Missing Atoms

X-ray crystallography structures rarely include hydrogen positions. Before running molecular dynamics simulations, hydrogen atoms must be added to satisfy valence and protonation states.

Let's repair our T4 Lysozyme system by adding all missing hydrogens using `msm.build.add_missing_hydrogens()`:

In [3]:
# Inspect initial atom count (heavy atoms only)
n_atoms_before = msm.get(lysozyme, element='system', n_atoms=True)
print(f"Atom count before adding hydrogens: {n_atoms_before}")

# Repair system by adding missing hydrogens
lysozyme_repaired = msm.build.add_missing_hydrogens(lysozyme)

# Inspect updated atom count
n_atoms_after = msm.get(lysozyme_repaired, element='system', n_atoms=True)
print(f"Atom count after adding hydrogens : {n_atoms_after}")
print(f"Has hydrogens after repair? {msm.build.has_hydrogens(lysozyme_repaired)}")

Atom count before adding hydrogens: 1441


Atom count after adding hydrogens : 3026
Has hydrogens after repair? True


### 3. Reconstructing Covalent Bonds

Standard PDB files often lack explicit CONECT records for protein backbones or side chains. You can audit missing covalent bonds using `msm.build.get_missing_bonds()` and reconstruct missing connectivity with `msm.build.add_missing_bonds()`:

In [4]:
# Audit missing covalent bonds in PDB 181L
missing_bonds = msm.build.get_missing_bonds('pdb_id:181L')
print(f"Missing covalent bonds in PDB 181L: {len(missing_bonds)}")

# Reconstruct missing covalent bonds into a native MolSys object
rebuilt_system = msm.convert('pdb_id:181L', to_form='molsysmt.MolSys')
msm.build.add_missing_bonds(rebuilt_system)

# Verify that missing bonds count is now 0
print(f"Missing covalent bonds after repair: {len(msm.build.get_missing_bonds(rebuilt_system))}")

Missing covalent bonds in PDB 181L: 1322


Missing covalent bonds after repair: 0


### 4. Building Peptides from Sequence

In addition to auditing and repairing experimental structures, `msm.build` can synthesize new peptide models directly from amino acid sequence strings using **`msm.build.build_peptide()`**:

In [5]:
# Build an acetylated/N-methyl capped Alanine dipeptide
peptide = msm.build.build_peptide('AceAlaNme')

# Inspect generated peptide attributes
n_atoms, n_groups = msm.get(peptide, n_atoms=True, n_groups=True)
print(f"Synthesized peptide: {n_groups} groups, {n_atoms} atoms")

Synthesized peptide: 3 groups, 22 atoms


:::{hint}
:class: dropdown
**msm.build.build_peptide()**: Synthesizes a 3D peptide structure from a 1-letter or 3-letter amino acid sequence code. See API doc: {func}`molsysmt.build.build_peptide`.
:::

--- 

### 🏆 Challenge 12: The Quality Auditor

1. Load the **T4 Lysozyme** structure using `'pdb_id:181L'`.
2. Check if it contains hydrogen atoms using `msm.build.has_hydrogens()`.
3. Convert it to `molsysmt.MolSys` and add missing hydrogens using `msm.build.add_missing_hydrogens()`.
4. Synthesize a tripeptide `'AceAlaAlaNme'` using `msm.build.build_peptide()`.

Auditing, repairing, and synthesizing systems ensures physical validity before simulation. In [Module 13: Topological Analysis](../Common_Core/13_Topological_Analysis.ipynb), we will explore non-spatial topological features.

```{key-takeaway}
The `msm.build` module provides tools for quality auditing (`get_non_standard_residues`, `get_missing_bonds`, `has_hydrogens`), structural repair (`add_missing_hydrogens`, `add_missing_bonds`), and system synthesis (`build_peptide`, `solvate`) before downstream calculations.
```

(course-core-12-see-also)=
:::{seealso}
:class: dropdown
**API Documentation for Functions in this Module:**
- {func}`molsysmt.build.get_non_standard_residues` — Non-standard group audit tool.
- {func}`molsysmt.build.add_missing_hydrogens` — Hydrogen addition and protonation engine.
- {func}`molsysmt.build.add_missing_bonds` — Covalent bond reconstruction engine.
- {func}`molsysmt.build.build_peptide` — Peptide synthesis engine.

**Related Course Modules & Guides:**
- Previous Module: [Module 11: Iterating Systems](../Common_Core/11_Iterating_Systems.ipynb)
- Next Module: [Module 13: Topological Analysis](../Common_Core/13_Topological_Analysis.ipynb)
- User Guide: {ref}`user-foundations`
:::